# Can a physical-AI model tell possible motion from impossible motion?

Question: can Cosmos Reason 2, a reasoning model built for physical AI, tell possible motion from impossible motion better than general vision models, and can it work as a critic that filters synthetic video?

Setup: a CPU notebook that calls the NVIDIA-hosted, OpenAI-compatible API (`https://integrate.api.nvidia.com/v1`) with Python standard library, numpy, PIL and pandas. It needs an `NVIDIA_API_KEY` secret. A small 2D physics simulation renders 24 clips of 3 seconds each. Twelve clips obey physics. Twelve break one physical rule. The simulator gives the exact answer for every clip. Each hosted model sees each clip once and answers one fixed question at temperature 0.

Terms used below:

- **Physical AI**: AI that senses, reasons about and acts in the physical world. Robots, vehicles and factories are examples.
- **World foundation model (WFM)**: a large model of how the physical world changes over time. NVIDIA Cosmos is a platform of WFMs.
- **Cosmos Predict, Transfer, Reason**: three Cosmos model types, split by job. Predict generates future video. Transfer restyles video from control maps. Reason is a vision-language model that reads video and writes text.
- **Vision-language model (VLM)**: a model that takes images or video plus text and writes text.
- **Omni model**: one model that takes several input types, such as text, image, video and audio.
- **Generative and discriminative**: a generative model creates new content, such as a video. A discriminative model gives a label to an input, such as plausible or impossible. A VLM writes text, but when you force a fixed label as the answer, you use it as a discriminative judge.
- **Synthetic data**: data made by a simulator or a generator, not collected in the real world.
- **Ground truth**: the known right label. A simulator gives it for free.
- **Critic**: a model that judges generated data and approves or rejects each item.
- **Purity and yield**: after filtering, purity is the share of kept clips that are truly good. Yield is the share of all clips that you keep.
- **Precision and recall for "impossible"**: precision is the share of "impossible" answers that are right. Recall is the share of impossible clips that the model finds.
- **Wilson interval**: a 95% range for a share. With 24 clips the range is wide, so a small gap between two models means little.
- **Reasoning format**: when the prompt asks for it, Cosmos Reason writes a `<think>` block first and the final answer after it.
- **Inline media**: the clip travels inside the request body as base64 text.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Question | What you measure | Your guess |
|---|---|---|
| Does Cosmos Reason 2 beat the always-plausible baseline of 0.5? | Accuracy and its 95% Wilson interval | |
| Does it beat the general vision models? | Accuracy, precision and recall for "impossible" | |
| Which models lean towards "plausible" no matter what they see? | Share of "plausible" answers | |
| Does the `<think>` reasoning format help Cosmos Reason? | Accuracy, tokens and latency, with and without it | |
| Which violation types are easy and which are hard? | Detection rate per violation type | |
| Does a model that finds impossible clips also say why? | Share of correct answers whose evidence names the right violation | |
| Can the best model act as a data critic? | Purity before and after, yield, false rejections | |
| Do the models accept a whole video, or only frames? | Probe table, per model | |

In [ ]:
# 1. Install: nothing for the API client. imageio-ffmpeg is a fallback video encoder if the image has no ffmpeg.
!pip install -q imageio-ffmpeg 2>&1 | tail -2
!ffmpeg -version 2>&1 | head -1

In [ ]:
# 2. Setup: constants and API key (standard library + numpy + PIL + pandas)
import base64, http.client, io, json, math, os, random, re, shutil, statistics, subprocess, time, urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageFont

BASE = os.environ.get("NIM_BASE_URL", "https://integrate.api.nvidia.com/v1")
COSMOS = "nvidia/cosmos-reason2-8b"
MODELS = {   # model id -> max_tokens for one reply. Reasoning models need room to think and then answer.
    COSMOS: 400,
    "nvidia/nemotron-3.5-super-vl-120b-a12b": 4000,
    "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning": 4000,
    "google/gemma-3-12b-it": 400,
    "google/gemma-3-4b-it": 400,
}
THINK_TOKENS = 2000   # Cosmos with the <think> format
SEED = 42
REQ_TIMEOUT_S, MAX_RETRIES, BACKOFF_S, RETRY_CAP_S = 180, 4, 2.0, 30.0
CONCURRENCY = 2
PROBE_BUDGET_S, CONFIG_BUDGET_S, TOTAL_BUDGET_S = 180, 600, 45 * 60
# NVIDIA documents 180 KB per inline image (base64 length) for hosted VLMs. No video limit is documented for this
# endpoint, so each clip stays under 130 KB (about 173 KB after base64) and the probe below shows what each model takes.
CLIP_LIMIT_B = 130_000
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def get_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("NVIDIA_API_KEY")
    except Exception:
        key = os.environ.get("NVIDIA_API_KEY")
    if not key:
        raise RuntimeError("NVIDIA_API_KEY not found. Add-ons > Secrets > add NVIDIA_API_KEY, then switch it on for this notebook.")
    return key

KEY = get_key()
print("key found:", bool(KEY))   # never print the key itself

In [ ]:
# 3. Helpers: one chat call with timeout, retry and backoff; parallel map; answer parser; Wilson interval
def chat(model, messages, extra=None, max_tokens=400, deadline=None):
    body = {"model": model, "messages": messages, "max_tokens": max_tokens, "temperature": 0, "seed": SEED, **(extra or {})}
    data = json.dumps(body).encode()
    out = dict(ok=False, error="", content="", reasoning="", finish="", prompt_tokens=0, completion_tokens=0,
               latency_s=float("nan"), retries=0)
    for attempt in range(MAX_RETRIES + 1):
        timeout = REQ_TIMEOUT_S if deadline is None else max(1, min(REQ_TIMEOUT_S, deadline - time.time()))
        req = urllib.request.Request(BASE + "/chat/completions", data,
                                     {"Authorization": "Bearer " + KEY, "Content-Type": "application/json"})
        t0, wait = time.time(), None
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                j = json.loads(r.read())
            ch = j["choices"][0]; m = ch["message"]; u = j.get("usage") or {}
            out.update(ok=True, error="", content=m.get("content") or "",
                       reasoning=m.get("reasoning_content") or m.get("reasoning") or "",
                       finish=ch.get("finish_reason") or "", prompt_tokens=u.get("prompt_tokens") or 0,
                       completion_tokens=u.get("completion_tokens") or 0, latency_s=time.time() - t0)
            return out
        except urllib.error.HTTPError as e:           # 429 and 5xx are retried, other codes are final
            out["error"] = f"HTTP {e.code}"
            if e.code != 429 and e.code < 500:
                return out
            try: wait = float(e.headers.get("Retry-After", ""))   # honour Retry-After
            except ValueError: wait = None
        except (ValueError, KeyError, IndexError, TypeError, AttributeError) as e:   # malformed reply
            out["error"] = "bad reply " + type(e).__name__
            return out
        except (OSError, http.client.HTTPException) as e:   # timeout, refused, reset, cut-off body
            out["error"] = type(getattr(e, "reason", e)).__name__
        if attempt == MAX_RETRIES:
            break
        wait = min(BACKOFF_S * 2 ** attempt if wait is None else wait, RETRY_CAP_S)
        if deadline is not None and time.time() + wait >= deadline:
            break
        time.sleep(wait); out["retries"] += 1
    return out

def pmap(fn, items, deadline):   # at most CONCURRENCY calls at once; items not started by the deadline give None
    with ThreadPoolExecutor(CONCURRENCY) as ex:
        return list(ex.map(lambda x: fn(x) if time.time() < deadline else None, items))

def hosted_models():
    try:
        with urllib.request.urlopen(BASE + "/models", timeout=30) as r:   # works without a key
            return {m["id"] for m in json.load(r)["data"]}
    except Exception as e:
        print("could not list /models (" + type(e).__name__ + "); will try every model")
        return set(MODELS)

def answer_text(text):
    """The final answer of a reply: after the last </think>, inside <answer> if present. Unfinished thinking gives ''."""
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[1]
    elif "<think>" in text:
        return ""
    m = re.search(r"<answer>(.*?)(?:</answer>|$)", text, re.S)
    return (m.group(1) if m else text).strip()

VERDICT = re.compile(r"(?:\bnot|n't) +(?:physically +)?(?:plausible|possible)\b|\b(?:implausible|impossible|unphysical)\b|\b(?:plausible|possible)\b")
def parse_verdict(text):
    """'plausible', 'impossible' or None (empty reply, unfinished thinking, or no verdict word). First verdict word wins."""
    a = answer_text(text).lower()
    m = VERDICT.search(a)
    if m: return "impossible" if m.group(0)[:2] in ("no", "n'", "im", "un") else "plausible"
    m = re.match(r"\W*(yes|no)\b", a)   # a short yes or no to "Is the motion physically possible?"
    return None if not m else "plausible" if m.group(1) == "yes" else "impossible"

def wilson(k, n, z=1.96):   # 95% interval for k successes in n tries
    if n == 0:
        return float("nan"), float("nan")
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def score(g, n_total):   # g: rows of one model and mode. Errors and unparsed replies are left out of accuracy.
    p = g[g.outcome.isin(["correct", "wrong"])]
    n, k = len(p), int((p.outcome == "correct").sum())
    said = int((p.verdict == "impossible").sum()); pos = int((p.label == "impossible").sum())
    tp = int(((p.verdict == "impossible") & (p.label == "impossible")).sum())
    neg = n - pos; lo, hi = wilson(k, n)
    answered = g[g.outcome != "error"]
    nan = float("nan")
    return dict(n=n, errors=int((g.outcome == "error").sum()), parse_fail=int((g.outcome == "unparsed").sum()),
                accuracy=k / n if n else nan, ci95=f"{lo:.2f}-{hi:.2f}", acc_all=k / n_total,
                precision_imp=tp / said if said else nan, recall_imp=tp / pos if pos else nan,
                false_alarm=(said - tp) / neg if neg else nan, plausible_share=(p.verdict == "plausible").mean() if n else nan,
                p50_s=answered.latency_s.median(), avg_tokens=answered.completion_tokens.mean())

## The clips

A small 2D simulation makes every clip, so the right answer is exact. Units are pixels and seconds. The picture is 320 by 240 pixels, 12 frames per second, 36 frames (3 seconds). Each clip has a black floor line, a timestamp at the top left and simple coloured shapes.

Five scene types, with a plausible clip and impossible variants:

| Scene | What happens in a plausible clip | Violation types used in impossible clips |
|---|---|---|
| `drop` | A ball falls and bounces. Each bounce is lower. | `gravity_reversal`, `energy_gain` |
| `roll` | A ball rolls off a table, falls and bounces. | `teleport`, `vanish_return`, `pass_through` |
| `pendulum` | A pendulum swings. Each swing is a little lower. | `amplitude_growth`, `vanish_return` |
| `collide` | A ball hits a resting ball. Momentum is conserved and energy does not grow. | `pass_through`, `spontaneous_motion`, `energy_gain` |
| `push` | A block hits a resting block. The pushed block slides and stops. | `spontaneous_motion`, `teleport` |

The 12 impossible clips hold seven violation types: `gravity_reversal` (1 clip), `pass_through` (2), `vanish_return` (2), `energy_gain` (2), `amplitude_growth` (1), `teleport` (2), `spontaneous_motion` (2). Most types have one or two clips, so a rate per type is a rough hint and not a measurement. Some violations break more than one rule. A teleport, for example, also adds energy. Each clip is named after its main violation.

Before any clip goes to a model, the next cell checks the ground truth from the simulated numbers. In plausible clips, energy never grows, nothing vanishes, jumps or sinks, and a resting object moves only after it is hit. In each impossible clip, the claimed rule really breaks at the claimed frame.

In [ ]:
# 4. Physics: five scenes, exact trajectories, rendering. Units: pixels and seconds, y points down.
W, H, FLOOR, G, RB = 320, 240, 190, 900.0, 12
FPS, NF, SUB = 12, 36, 40                 # 36 frames = 3 s, 40 physics steps per frame
DT = 1 / (FPS * SUB)
TABLE_X, TABLE_Y = 150, 110
TOP = FLOOR - RB                          # centre height of a ball that rests on the floor
RED, GREEN, BLUE, ORANGE, BROWN, GRAY, DARK = (215, 40, 40), (30, 150, 60), (40, 80, 210), (235, 140, 20), (125, 85, 50), (140, 140, 140), (30, 30, 30)
frame_of = lambda k: -(-(k + 1) // SUB)   # first frame that shows the effect of physics step k

def new(scene, viol):
    return dict(scene=scene, viol=viol, label="impossible" if viol else "plausible", vf=None, nv=0,
                E=[], ang=[], pos={}, vis={}, shapes=[], half={})

def snap(c, shapes, **bodies):            # one frame: shapes to draw, and name -> (x, y, visible) for the checks
    c["shapes"].append([s for s in shapes if s])
    for n, (x, y, v) in bodies.items():
        c["pos"].setdefault(n, []).append((x, y)); c["vis"].setdefault(n, []).append(v)

def ball_step(b, g, e, solid):
    """Exact constant-acceleration step with a floor bounce of restitution e. Returns True when the ball reaches the floor."""
    y1 = b["y"] + b["vy"] * DT + 0.5 * g * DT * DT
    touch = g > 0 and b["y"] <= TOP <= y1
    if touch and solid:   # impact speed from energy, so a bounce with e <= 1 never adds energy
        b["y"], b["vy"] = TOP, -e * (b["vy"] ** 2 + 2 * g * (TOP - b["y"])) ** 0.5
    else:
        b["y"], b["vy"] = y1, b["vy"] + g * DT
    b["x"] += b["vx"] * DT
    if not RB <= b["x"] <= W - RB:        # the frame edge acts as a wall
        b["vx"] = -b["vx"]; b["x"] = min(max(b["x"], RB), W - RB)
    return touch

def sim_ball(rng, scene, viol):           # scenes drop and roll
    c, roll = new(scene, viol), scene == "roll"
    b = dict(x=rng.uniform(40, 70), y=TABLE_Y - RB, vx=rng.uniform(80, 110), vy=0.0) if roll else \
        dict(x=rng.uniform(80, 240), y=rng.uniform(30, 50), vx=rng.uniform(-30, 30), vy=0.0)
    e, g, leave = rng.uniform(0.55, 0.75), G, None
    if viol == "energy_gain":
        e, b["y"] = 1.15, 100.0
    for k in range(NF * SUB):
        if k % SUB == 0:
            hide = viol == "vanish_return" and leave is not None and c["vf"] <= k // SUB < c["vf"] + c["nv"]
            snap(c, ([("r", BROWN, 0, TABLE_Y, TABLE_X, FLOOR)] if roll else []) + [None if hide else ("c", RED, b["x"], b["y"], RB)],
                 ball=(b["x"], b["y"], not hide))
        if viol == "gravity_reversal" and k == 4 * SUB:
            g, c["vf"] = -G, frame_of(k)
        if roll and leave is None:        # rolls on the table and slows a little
            b["vx"] = max(0.0, b["vx"] - 10 * DT); b["x"] += b["vx"] * DT
            if b["x"] >= TABLE_X:
                leave = k
                if viol == "vanish_return":
                    c["vf"], c["nv"] = frame_of(k) + 2, 5
        else:
            if viol == "teleport" and k == leave + 82:
                b["x"] += 60; b["y"] -= 70; c["vf"] = frame_of(k)
            if ball_step(b, g, e, viol != "pass_through") and c["vf"] is None and viol in ("pass_through", "energy_gain"):
                c["vf"] = frame_of(k)
        c["E"].append(0.5 * (b["vx"] ** 2 + b["vy"] ** 2) + G * (TOP - b["y"]))
    return c

def rk4(f, s, h):
    a = f(s); b = f(s + h / 2 * a); c = f(s + h / 2 * b); d = f(s + h * c)
    return s + h / 6 * (a + 2 * b + 2 * c + d)

def sim_pendulum(rng, scene, viol):
    c, L, px, py = new(scene, viol), 70.0, W / 2, 50
    damp = -0.5 if viol == "amplitude_growth" else rng.uniform(0.1, 0.3)   # negative damping adds energy
    a0 = rng.uniform(0.45, 0.6) if viol == "amplitude_growth" else rng.uniform(0.5, 0.9)
    s = np.array([rng.choice([-1, 1]) * a0, 0.0])                          # angle and angular speed
    if viol == "vanish_return":
        c["vf"], c["nv"] = 14, 5
    acc = lambda s: np.array([s[1], -G / L * math.sin(s[0]) - damp * s[1]])
    for k in range(NF * SUB):
        if k % SUB == 0:
            hide = viol == "vanish_return" and c["vf"] <= k // SUB < c["vf"] + c["nv"]
            bx, by = px + L * math.sin(s[0]), py + L * math.cos(s[0])
            snap(c, [("r", GRAY, px - 40, py - 8, px + 40, py), None if hide else ("l", DARK, px, py, bx, by),
                     None if hide else ("c", RED, bx, by, RB)], bob=(bx, by, not hide))
            c["ang"].append(s[0])
        s = rk4(acc, s, DT)
        c["E"].append(0.5 * L * L * s[1] ** 2 + G * L * (1 - math.cos(s[0])))
    return c

def sim_pair(rng, scene, viol):           # scenes collide (balls) and push (blocks): A moves right and hits resting B
    c, push = new(scene, viol), scene == "push"
    m = [1.0, rng.uniform(0.6, 1.8)]
    h = [20.0, 20.0 * m[1] ** 0.5] if push else [RB, RB * m[1] ** 0.5]     # half widths. A heavier B is bigger.
    mu = 60.0 if push else 80.0                                            # floor friction (deceleration)
    e = 1.6 if viol == "energy_gain" else rng.uniform(0.1, 0.3) if push else rng.uniform(0.7, 1.0)
    x = [rng.uniform(40, 50), rng.uniform(170, 190)] if push else [rng.uniform(30, 40), rng.uniform(160, 170)]
    v = [rng.uniform(150, 190), 0.0]
    col = (BLUE, ORANGE) if push else (RED, GREEN)
    c["half"] = dict(A=h[0], B=h[1])
    shape = lambda i: ("r", col[i], x[i] - h[i], FLOOR - 2 * h[i], x[i] + h[i], FLOOR) if push else ("c", col[i], x[i], FLOOR - h[i], h[i])
    for k in range(NF * SUB):
        if k % SUB == 0:
            snap(c, [shape(0), shape(1)], A=(x[0], FLOOR - h[0], True), B=(x[1], FLOOR - h[1], True))
        if viol == "spontaneous_motion" and k == 3 * SUB:
            v[1], c["vf"] = 60.0 if push else 70.0, frame_of(k)             # B starts to move, nothing touches it
        if viol == "teleport" and k == 4 * SUB:
            x[1] += 80; c["vf"] = frame_of(k)
        if x[1] - x[0] <= h[0] + h[1] and v[0] > v[1]:                      # contact
            if c["vf"] is None and viol in ("pass_through", "energy_gain"):
                c["vf"] = frame_of(k)
            if viol != "pass_through":                                      # 1D collision with restitution e
                v = [((m[0] - e * m[1]) * v[0] + (1 + e) * m[1] * v[1]) / (m[0] + m[1]),
                     ((m[1] - e * m[0]) * v[1] + (1 + e) * m[0] * v[0]) / (m[0] + m[1])]
        for i in (0, 1):
            v[i] = math.copysign(max(0.0, abs(v[i]) - mu * DT), v[i]); x[i] += v[i] * DT
        c["E"].append(sum(0.5 * m[i] * v[i] ** 2 for i in (0, 1)))
    return c

try: FONT = ImageFont.load_default(size=14)
except TypeError: FONT = ImageFont.load_default()

def render(shapes, f):                    # draw at 2x and shrink, so the edges are smooth
    S = 2
    im = Image.new("RGB", (W * S, H * S), (245, 245, 245)); d = ImageDraw.Draw(im)
    d.rectangle([0, FLOOR * S, W * S, H * S], fill=(205, 205, 205))
    d.line([0, FLOOR * S, W * S, FLOOR * S], fill=(0, 0, 0), width=3 * S)
    for kind, colour, *a in shapes:
        a = [t * S for t in a]
        if kind == "c": d.ellipse([a[0] - a[2], a[1] - a[2], a[0] + a[2], a[1] + a[2]], fill=colour, outline=(0, 0, 0), width=S)
        elif kind == "r": d.rectangle(a, fill=colour, outline=(0, 0, 0), width=S)
        else: d.line(a, fill=colour, width=3 * S)
    im = im.resize((W, H), Image.LANCZOS)
    ImageDraw.Draw(im).text((6, 5), f"t={f / FPS:.2f}s", fill=(0, 0, 0), font=FONT)
    return im

def check(c):
    """Ground truth from the simulated numbers. Plausible clips keep every rule. Impossible clips break the claimed rule at the claimed frame."""
    E, f, v, scene = np.array(c["E"]), c["vf"], c["viol"], c["scene"]
    P = {n: np.array(a) for n, a in c["pos"].items()}
    step = lambda n: np.linalg.norm(np.diff(P[n], axis=0), axis=1)     # pixels moved between two frames
    ang = np.abs(c["ang"]); gap = None
    if "B" in P:
        gap = np.abs(P["B"][:, 0] - P["A"][:, 0]) - sum(c["half"].values())   # free space between A and B
    if not v:
        assert (np.diff(E) <= 1e-6 * E[0]).all(), "energy grew"
        assert all(all(x) for x in c["vis"].values()), "an object vanished"
        assert all(step(n).max() < 50 for n in P), "an object jumped"
        if "ball" in P: assert P["ball"][:, 1].max() <= TOP + 1e-6, "a ball sank into the floor"
        if gap is not None:
            moved = np.abs(P["B"][:, 0] - P["B"][0, 0]) > 0.5
            assert gap.min() > -1.5 and moved.any() and gap[moved.argmax() - 1] < 20, "B overlaps A or moved without a hit"
        if scene == "pendulum": assert ang[-12:].max() <= ang[:6].max(), "the swing grew"
        return
    if v == "gravity_reversal":
        y = P["ball"][:, 1]; d2 = lambda i: y[i + 1] - 2 * y[i] + y[i - 1]      # second difference is the acceleration
        assert d2(f - 2) > 4 and all(d2(i) < -4 for i in (f, f + 1, f + 2)) and y[f:f + 4].max() < TOP - 10, "gravity did not flip in mid-air"
    elif v == "energy_gain":
        dE = np.diff(E); i = int(np.argmax(dE > 0.05 * E[0]))
        assert dE[i] > 0.05 * E[0] and frame_of(i + 1) == f, "energy did not jump at the claimed frame"
    elif v == "pass_through" and scene == "collide":
        assert gap[f:].min() < -0.5 * sum(c["half"].values()), "the balls did not overlap"
    elif v == "pass_through":
        y = P["ball"][:, 1]; assert y[f - 1] <= TOP + 1e-6 and y[f + 2] > TOP + 20, "the ball did not sink into the floor"
    elif v == "vanish_return":
        h = np.array(c["vis"]["bob" if scene == "pendulum" else "ball"])
        assert c["nv"] >= 3 and h[f - 1] and not h[f:f + c["nv"]].any() and h[f + c["nv"]], "no vanish and return"
    elif v == "teleport":
        s = step("ball" if "ball" in P else "B"); assert s[f - 1] > 70 and np.delete(s, f - 1).max() < 50, "no single-frame jump"
    elif v == "amplitude_growth":
        assert ang[-12:].max() > 1.4 * ang[:6].max(), "the swing did not grow"
    elif v == "spontaneous_motion":
        assert gap[f - 1] > 20 and abs(P["B"][f - 1, 0] - P["B"][0, 0]) < 0.5 and abs(P["B"][f + 2, 0] - P["B"][f - 1, 0]) > 8, "B did not start moving alone"
    else:
        raise AssertionError("unknown violation " + v)

In [ ]:
# 5. Build 24 clips (12 plausible, 12 impossible), check the ground truth, encode to mp4, make frame images, show a contact sheet
PLAN = [("drop", ""), ("drop", ""), ("drop", ""), ("drop", "gravity_reversal"), ("drop", "energy_gain"),
        ("roll", ""), ("roll", ""), ("roll", "teleport"), ("roll", "vanish_return"), ("roll", "pass_through"),
        ("pendulum", ""), ("pendulum", ""), ("pendulum", ""), ("pendulum", "amplitude_growth"), ("pendulum", "vanish_return"),
        ("collide", ""), ("collide", ""), ("collide", "pass_through"), ("collide", "spontaneous_motion"), ("collide", "energy_gain"),
        ("push", ""), ("push", ""), ("push", "spontaneous_motion"), ("push", "teleport")]
SIM = dict(drop=sim_ball, roll=sim_ball, pendulum=sim_pendulum, collide=sim_pair, push=sim_pair)
IMG_IDX = np.linspace(0, NF - 1, 6).round().astype(int)   # 6 frames for models that do not take video
SHEET_IDX = [0, 6, 12, 18, 24, 30]

def find_encoder():
    exe = shutil.which("ffmpeg")
    if not exe:
        try:
            import imageio_ffmpeg; exe = imageio_ffmpeg.get_ffmpeg_exe()
        except Exception:
            pass
    if exe: return "ffmpeg", exe
    try:
        import cv2; return "cv2", None
    except ImportError:
        return None, None

def encode_mp4(frames, path, crf):
    kind, exe = ENCODER
    if kind == "ffmpeg":
        for codec in ("libx264", "mpeg4"):   # H.264 first. MPEG-4 part 2 if this ffmpeg build has no libx264.
            cmd = [exe, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}", "-r", str(FPS), "-i", "-",
                   "-c:v", codec, "-pix_fmt", "yuv420p", "-movflags", "+faststart", path]
            cmd[-1:-1] = ["-crf", str(crf), "-preset", "veryfast"] if codec == "libx264" else ["-q:v", str(crf // 2)]
            if subprocess.run(cmd, input=b"".join(f.tobytes() for f in frames), capture_output=True).returncode == 0:
                return
        raise RuntimeError("ffmpeg could not write an mp4")
    import cv2
    vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
    for f in frames: vw.write(cv2.cvtColor(np.asarray(f), cv2.COLOR_RGB2BGR))
    vw.release()

def jpeg_b64(im):
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=75); return base64.b64encode(buf.getvalue()).decode()

ENCODER = find_encoder()
print("video encoder:", ENCODER[0] or "none (frames only)")
os.makedirs(f"{OUT_DIR}/clips", exist_ok=True)
CLIPS, seen = [], {}
for scene, viol in PLAN:
    i = seen[scene] = seen.get(scene, -1) + 1
    c = SIM[scene](random.Random(f"{SEED}-{scene}-{i}"), scene, viol)
    c["id"] = f"{scene}-{i}"
    check(c)
    frames = [render(s, f) for f, s in enumerate(c.pop("shapes"))]
    assert len(frames) == NF
    c["b64_video"], c["mp4_kb"] = "", float("nan")
    if ENCODER[0]:
        path = f"{OUT_DIR}/clips/{c['id']}.mp4"
        for crf in (28, 34, 40):          # a higher crf gives a smaller file
            encode_mp4(frames, path, crf)
            if os.path.getsize(path) <= CLIP_LIMIT_B: break
        assert os.path.getsize(path) <= CLIP_LIMIT_B, "clip too large for inline media"
        c["b64_video"], c["mp4_kb"] = base64.b64encode(open(path, "rb").read()).decode(), os.path.getsize(path) / 1000
    c["b64_frames"] = [jpeg_b64(frames[j]) for j in IMG_IDX]
    c["thumbs"] = [frames[j] for j in SHEET_IDX]
    CLIPS.append(c)
VIDEO_OK = bool(ENCODER[0])
assert len(CLIPS) == 24 and sum(c["label"] == "plausible" for c in CLIPS) == 12
INFO = pd.DataFrame([dict(id=c["id"], label=c["label"], violation=c["viol"], claimed_frame=c["vf"], mp4_kb=round(c["mp4_kb"], 1),
                          frames_kb=round(sum(len(b) for b in c["b64_frames"]) * 0.75 / 1000, 1)) for c in CLIPS])
INFO.to_csv(f"{OUT_DIR}/cosmos_physical_check_clips.csv", index=False)
print(INFO.to_string(index=False))
print("all ground-truth checks passed for", len(CLIPS), "clips")

pick = [CLIPS[0]] + [next(c for c in CLIPS if (c["scene"], c["viol"]) == sv) for sv in
        [("drop", "gravity_reversal"), ("roll", "teleport"), ("pendulum", "amplitude_growth"), ("collide", "pass_through"), ("push", "spontaneous_motion")]]
tw, th = W // 2, H // 2
sheet = Image.new("RGB", (tw * len(SHEET_IDX), (th + 16) * len(pick)), "white"); d = ImageDraw.Draw(sheet)
for r, c in enumerate(pick):
    d.text((4, r * (th + 16) + 1), f"{c['id']}: {c['label']} {c['viol']}", fill=(0, 0, 0))
    for j, im in enumerate(c["thumbs"]):
        sheet.paste(im.resize((tw, th)), (j * tw, r * (th + 16) + 16))
sheet.save(f"{OUT_DIR}/cosmos_physical_check_contact_sheet.png")
try:
    from IPython.display import display; display(sheet)
except ImportError:
    pass

## Media probe: video or frames?

The hosted API takes a video as a `video_url` content part with base64 data (`data:video/mp4;base64,...`). The media part must come before the text part. The Cosmos NIM docs also list `media_io_kwargs` to set the sampling rate (frames per second). Not every hosted model takes video, and this notebook does not assume which ones do.

The next cell sends one clip to each model in up to three forms and keeps the first form that returns a normal reply:

| Form | What is sent |
|---|---|
| `video+fps` | The mp4 clip with `media_io_kwargs: {"video": {"fps": 6.0}}` |
| `video` | The mp4 clip only |
| `frames` | Six frames of the clip as `image_url` parts, in time order |

A form counts as working when the API returns a reply that is not empty. The probe cannot tell if a model really used all of the video. A model that is missing from the hosted list, or that fails every form, is skipped and the cell prints why.

About the size limit: the NVIDIA API reference for hosted VLMs documents 180 KB per inline image (the length of the base64 text) and an upload API for larger images. I found no documented limit for inline video on this endpoint. So each clip stays under 130 KB, which is about 173 KB after base64. The table at the end of the clip cell shows the real sizes.

In [ ]:
# 6. Prompt, request builder and media probe: one clip per model and form
QUESTION = "Is the motion physically possible? Answer plausible or impossible and name the evidence."
THINK = "\n\nAnswer the question using the following format:\n\n<think>\nYour reasoning.\n</think>\n\nWrite your final answer immediately after the </think> tag."
FORMS = {"video+fps": {"media_io_kwargs": {"video": {"fps": 6.0}}}, "video": {}, "frames": {}}

def request(clip, form, think=False):
    """Messages and extra body fields for one clip. The media part comes before the text part."""
    if form.startswith("video"):
        media, lead = [{"type": "video_url", "video_url": {"url": "data:video/mp4;base64," + clip["b64_video"]}}], ""
    else:
        media = [{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + b}} for b in clip["b64_frames"]]
        lead = "These images are frames of one video, in time order. "
    text = lead + QUESTION + (THINK if think else "")
    return [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": media + [{"type": "text", "text": text}]}], FORMS[form]

START = time.time(); END = START + TOTAL_BUDGET_S
HOSTED = hosted_models()
USE = [m for m in MODELS if m in HOSTED]
for m in MODELS:
    if m not in HOSTED: print("skip (not in /v1/models):", m)
PROBE_LOG = []

def probe(model):
    for form in [f for f in FORMS if VIDEO_OK or f == "frames"]:
        msgs, extra = request(CLIPS[0], form)
        r = chat(model, msgs, extra, max_tokens=300, deadline=START + PROBE_BUDGET_S)
        works = r["ok"] and bool((r["content"] + r["reasoning"]).strip())
        PROBE_LOG.append(dict(model=model.split("/")[-1], form=form, works=works, error=r["error"], finish=r["finish"], latency_s=round(r["latency_s"], 1)))
        if works: return form

FORM = dict(zip(USE, pmap(probe, USE, START + PROBE_BUDGET_S)))
print(pd.DataFrame(PROBE_LOG).to_string(index=False))
for m in USE:
    print(f"{m.split('/')[-1]:42s}", FORM[m] or "no form worked: skipped")

## Runner

Each model and mode is one config. A config sends the 24 clips in a fixed mixed order, two requests at a time, with a time budget. If the budget ends first, the config keeps the results it has and the tables show how many clips finished (`n`). Cosmos Reason runs twice: `plain` (the question only) and `think` (the question plus the reasoning format from the Cosmos prompt guide). The other models run as they ship (`plain`) and with reasoning switched off (`off`).

Each reply gets one outcome: `correct`, `wrong`, `unparsed` (empty reply, thinking cut off by the token limit, or no verdict word) or `error` (the API call failed after retries). Accuracy uses `correct` and `wrong` only. The column `acc_all` counts `unparsed` and `error` as wrong, over all 24 clips. The prompt names no violation type and gives no hint about the variants.

In [ ]:
# 7. Run every config: Cosmos plain and with <think>; each other model as it ships and with reasoning off
ORDER = random.Random(SEED).sample(range(len(CLIPS)), len(CLIPS))   # mixed scenes, so a cut-off run still covers all of them
ROWS = []

def run_config(model, mode, think):
    form, max_tokens = FORM[model], THINK_TOKENS if think else MODELS[model]
    deadline = min(time.time() + CONFIG_BUDGET_S, END)
    t0, n0, st = time.time(), len(ROWS), {"ok": 0, "err": 0}
    def one(i):
        if st["ok"] == 0 and st["err"] >= 4: return   # the model fails every call: stop early
        c = CLIPS[i]; msgs, extra = request(c, form, think)
        if mode == "off": extra = dict(extra, chat_template_kwargs={"enable_thinking": False})
        r = chat(model, msgs, extra, max_tokens=max_tokens, deadline=deadline)
        v = parse_verdict(r["content"]) if r["ok"] else None
        out = "error" if not r["ok"] else "unparsed" if v is None else "correct" if v == c["label"] else "wrong"
        st["ok" if r["ok"] else "err"] += 1
        ROWS.append(dict(cfg=model.split("/")[-1] + "/" + mode, model=model, mode=mode, form=form, clip=c["id"], scene=c["scene"],
                         label=c["label"], viol=c["viol"], verdict=v, outcome=out, evidence=answer_text(r["content"])[:400],
                         finish=r["finish"], latency_s=r["latency_s"], completion_tokens=r["completion_tokens"],
                         retries=r["retries"], error=r["error"]))
    pmap(one, ORDER, deadline)
    mine = ROWS[n0:]
    note = "" if len(mine) == len(CLIPS) else f"  (cut off: {len(mine)} of {len(CLIPS)} clips reached)"
    print(f"{model.split('/')[-1]:42s} {mode:6s} {form:10s} correct {sum(x['outcome'] == 'correct' for x in mine):2d}  "
          f"unparsed {sum(x['outcome'] == 'unparsed' for x in mine):2d}  errors {sum(x['outcome'] == 'error' for x in mine):2d}  {time.time() - t0:5.0f}s{note}")
    if mine and all(x["outcome"] == "error" for x in mine):
        print("   skipped: every request failed, last error:", mine[-1]["error"])

for m in USE:
    if FORM.get(m):
        run_config(m, "plain", False)
        run_config(m, "think", True) if m == COSMOS else run_config(m, "off", False)
print(f"total {(time.time() - START) / 60:.1f} min")

In [ ]:
# 8. Summary table per model and mode (also written to CSV), with the trivial baseline and 95% Wilson intervals
assert ROWS, "No model answered. Check the key, the model list and the probe table above."
RAW = pd.DataFrame(ROWS)
RAW.to_csv(f"{OUT_DIR}/cosmos_physical_check_raw.csv", index=False)
lo, hi = wilson(12, 24)
SUMMARY = pd.DataFrame(
    [dict(cfg=cfg, model=g.model.iloc[0], mode=g["mode"].iloc[0], **score(g, len(CLIPS))) for cfg, g in RAW.groupby("cfg", sort=False)]
    + [dict(cfg="always plausible", model="always plausible", mode="baseline", n=24, errors=0, parse_fail=0, accuracy=0.5, ci95=f"{lo:.2f}-{hi:.2f}",
            acc_all=0.5, precision_imp=float("nan"), recall_imp=0.0, false_alarm=0.0, plausible_share=1.0, p50_s=float("nan"), avg_tokens=float("nan"))])
SUMMARY.round(3).to_csv(f"{OUT_DIR}/cosmos_physical_check.csv", index=False)
print(SUMMARY.drop(columns=["model", "mode"]).round(2).to_string(index=False))

# detection rate per violation type: impossible clips that the model called impossible, as hits/clips
imp = RAW[(RAW.label == "impossible") & RAW.outcome.isin(["correct", "wrong"])]
DETECT = imp.assign(hit=imp.verdict == "impossible").groupby(["viol", "cfg"]).hit.agg(lambda x: f"{int(x.sum())}/{len(x)}").unstack("cfg").fillna("-")
DETECT.to_csv(f"{OUT_DIR}/cosmos_physical_check_by_violation.csv")
print("\nImpossible clips found, per violation type (hits/clips):")
print(DETECT.to_string())

## Critic: filter a synthetic set

A data pipeline that makes video with a generator will keep some bad clips. Here the next cell builds a synthetic set with a known share of bad clips: 40 clips, 30% impossible. It draws 28 plausible and 12 impossible clips from the 24, with replacement, with a fixed seed. Then the best config from the summary table (highest `acc_all`) acts as the critic. A clip stays if the critic says `plausible`. A clip goes if the critic says `impossible`, or if it gave no usable answer.

The cell reports purity before and after, yield, and false rejections (good clips that the critic threw away). The same 24 clips repeat in the set, so their verdicts repeat too. The cell shows the filter arithmetic. It adds no new evidence about the model.

A filter is not a fix. It removes bad clips and does not repair them. Every false rejection is a good clip that you lose. Every miss is a bad clip that stays in the data.

In [ ]:
# 9. Critic: filter a synthetic set with a known 30% of bad clips, using the best config's verdicts
cand = SUMMARY[(SUMMARY["mode"] != "baseline") & (SUMMARY.n >= 12)]
if cand.empty:
    print("No config answered at least 12 clips, so there is no critic to test.")
else:
    best = cand.sort_values(["acc_all", "accuracy", "recall_imp"], ascending=False).iloc[0]   # acc_all counts unanswered clips as wrong
    verdict = RAW[RAW.cfg == best.cfg].set_index("clip").verdict.to_dict()   # clips with no usable answer are missing or None
    rng = random.Random(SEED)
    good, bad = [c["id"] for c in CLIPS if c["label"] == "plausible"], [c["id"] for c in CLIPS if c["label"] == "impossible"]
    pool = [rng.choice(good) for _ in range(28)] + [rng.choice(bad) for _ in range(12)]
    kept = [c for c in pool if verdict.get(c) == "plausible"]
    n_good_kept = sum(c in good for c in kept)
    CRITIC = dict(critic=best.cfg, set_size=len(pool), bad_share=sum(c in bad for c in pool) / len(pool),
                  purity_before=1 - sum(c in bad for c in pool) / len(pool),
                  purity_after=n_good_kept / len(kept) if kept else float("nan"), yield_kept=len(kept) / len(pool),
                  good_rejected=28 - n_good_kept, false_rejection_rate=(28 - n_good_kept) / 28,
                  bad_kept=len(kept) - n_good_kept, bad_removed_rate=1 - (len(kept) - n_good_kept) / 12)
    pd.DataFrame([CRITIC]).round(3).to_csv(f"{OUT_DIR}/cosmos_physical_check_critic.csv", index=False)
    print(pd.Series(CRITIC).to_string())
    print("This is a filter, not a fix: rejected clips are lost, not repaired, and every kept bad clip is a miss.")

## Grounding check: does the evidence name the right violation?

A correct "impossible" answer can still be a guess. The next cell looks at the answers where a model said "impossible" for a clip that really is impossible. It reads the stated evidence (the final answer, not the thinking block) and looks for words that fit the true violation type. For example `teleport` looks for words such as "jump", "suddenly" and "instant". The word lists are short and made by hand. This is a rough check. A hit does not prove that the model understood the clip, and a miss can be a good answer in other words. The column `names_any_violation` shows how often a reply hits any list at all, so you can see how loose the rubric is.

In [ ]:
# 10. Grounding check: keyword rubric on the stated evidence of correct "impossible" answers (rough check)
KEYS = {
    "gravity_reversal": ["gravity", "upward", "rises", "rising", "float", "against", "reverse", "fly", "flies", "defies"],
    "pass_through": ["through", "penetrat", "overlap", "phase", "sink", "sinks", "ghost", "clip", "no collision", "without colliding", "without bouncing"],
    "vanish_return": ["vanish", "disappear", "reappear", "invisible", "missing", "gone", "permanence", "pops"],
    "energy_gain": ["energy", "higher than", "faster than", "gains", "gain speed", "speeds up", "more speed", "higher each", "higher after"],
    "amplitude_growth": ["amplitude", "swing", "larger", "grow", "increas", "higher each", "wider"],
    "teleport": ["teleport", "jump", "suddenly", "instant", "discontinu", "skip", "abrupt"],
    "spontaneous_motion": ["no cause", "without", "spontaneous", "on its own", "by itself", "untouched", "no contact", "not touched", "no force", "nothing"],
}
hit = lambda text, t: any(w in text.lower() for w in KEYS[t])
gr = RAW[(RAW.label == "impossible") & (RAW.verdict == "impossible")]
gr = gr.assign(right=[hit(e, t) for e, t in zip(gr.evidence, gr.viol)], any_=[any(hit(e, t) for t in KEYS) for e in gr.evidence])
GROUND = gr.groupby("cfg").agg(correct_impossible=("clip", "size"), names_right_violation=("right", "sum"), names_any_violation=("any_", "sum"))
GROUND["right_rate"] = GROUND.names_right_violation / GROUND.correct_impossible
GROUND.round(2).to_csv(f"{OUT_DIR}/cosmos_physical_check_grounding.csv")
print(GROUND.round(2).to_string() if len(GROUND) else "No model gave a correct 'impossible' answer, so there is no evidence to check.")
for _, r in gr.head(6).iterrows():
    print(f"[{r.cfg} | {r.viol} | right={r.right}]", r.evidence[:160].replace("\n", " "))

## Cosmos and GR00T: what each takes in and gives out

The facts below come from the saved NVIDIA readings (Cosmos platform page, GR00T page) and from the NVIDIA documentation and blog pages listed in the `source` column. Cosmos 3 puts perception and generation in one omni model. The saved Cosmos page says Cosmos 2 and 2.5 kept them in separate models. NVIDIA still describes the Predict, Transfer and Reason roles in its Cosmos material, so they are a good way to learn what each part does.

In [ ]:
# 11. Facts table: Cosmos model types and Isaac GR00T
FACTS = pd.DataFrame([
    dict(family="Cosmos Predict", job="Generates future world states as video", takes_in="Text, image or video (also start and end frames)",
         gives_out="Video", used_here="no",
         source="NVIDIA technical blog 'Scale Synthetic Data and Physical AI Reasoning with NVIDIA Cosmos World Foundation Models'; NIM for Cosmos WFM docs (text2video, image2video, video2video)"),
    dict(family="Cosmos Transfer", job="Restyles a video and keeps its structure (for example sim to real)", takes_in="Control video (edge, blur, depth, segmentation or world-space map) plus a text prompt",
         gives_out="Photorealistic video with the same layout and motion", used_here="no",
         source="NIM for Cosmos WFM docs, Transfer page; saved reading cosmos.txt (Omniverse simulation into Cosmos Transfer)"),
    dict(family="Cosmos Reason", job="Reasons about physical scenes. Reads video and writes text. Can act as a critic.", takes_in="Video or image plus a text question",
         gives_out="Text, with an optional <think> block before the answer", used_here="yes" if FORM.get(COSMOS) else "no (no request worked, see the probe)",
         source="NIM for VLMs docs, Cosmos Reason2 API page (video_url, <think> format, Synthetic Data Generation Critic); Cosmos Cookbook prompt guide"),
    dict(family="Isaac GR00T", job="Robot policy. A vision-language-action (VLA) model for humanoid and other robots", takes_in="Camera views, a language instruction and the robot's current state",
         gives_out="Chunks of continuous robot actions", used_here="no",
         source="saved reading gr00t.txt (GR00T 1.7 is an open 3B VLA with a Cosmos backend)"),
])
FACTS.to_csv(f"{OUT_DIR}/cosmos_physical_check_facts.csv", index=False)
for _, r in FACTS.iterrows():
    print(f"{r.family} (used here: {r.used_here})\n  job:       {r.job}\n  takes in:  {r.takes_in}\n  gives out: {r.gives_out}\n  source:    {r.source}\n")

## What to look for

- **Baseline and intervals**: the always-plausible baseline scores 0.5. A model is only clearly better when its whole 95% Wilson interval sits above 0.5. With 24 clips, a gap of two or three clips between two models is inside the noise.
- **`plausible_share`**: a share near 1 means the model says "plausible" for almost everything. Its accuracy then tracks the baseline. A share near 0 means it flags almost everything.
- **`precision_imp`, `recall_imp`, `false_alarm`**: a critic that rejects too much has high recall and a high false alarm rate. A critic that approves too much has low recall.
- **`parse_fail`, `errors`, `acc_all`**: read accuracy together with these. A model with many unparsed replies can look good on the few clips it answered. `acc_all` counts those clips as wrong.
- **Cosmos `plain` against `think`**: compare accuracy with `avg_tokens` and `p50_s`. Reasoning costs output tokens and seconds. Check if the gain pays for them.
- **Detection per violation type**: most types have one or two clips. Look for patterns, such as motion that starts without a cause against a bounce with too much energy. Do not read one clip as a rate.
- **Grounding**: compare `names_right_violation` with `names_any_violation`. If the second is close to the first, the word lists are loose and the check says little.
- **Critic**: compare `purity_after` with `purity_before`, then look at `yield_kept` and `good_rejected`. A higher purity that costs most of the good clips may not be worth it.
- **Probe table**: a model that needs `frames` saw six still images, not a video, so it had less motion information than a model that took the video.
- **Limits**: the clips are simple 2D shapes, not real video. The question is fixed. One run at temperature 0 gives one answer per clip.

## Exam rules this proves

1. **physical.3**: Cosmos has model types split by job. Predict makes future video, Transfer restyles video from control maps, and Reason is a vision-language model that reads video and writes text. Isaac GR00T is a different kind of model: a vision-language-action policy that outputs robot actions.
2. **models.4**: A generative model creates content, such as video. A discriminative model gives a label. Cosmos Reason writes text, but with a fixed answer set it works as a discriminative judge. Check that it answers in the fixed set.
3. **physical.2**: A physics simulator gives exact ground truth for synthetic data. Use that truth to measure a critic, and check the truth before you trust it.
4. **physical.2**: A critic filters synthetic data. It removes bad items, so purity can rise and yield falls. It does not repair a bad item, and it can reject good items.
5. **models.4**: The Cosmos Reason `<think>` format is a prompt setting. It adds output tokens and latency, so test it against a plain prompt before you pay for it.